# Two views of event fidelity
Read saved EDS test scores. Frame stride changes RGB cadence on a shared scoring interval. K changes the number of consecutive input frames and the duration scored. The two figures use separate axes and comparisons.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments").is_dir())
REPORT = ROOT / "reports/experiments/eds/test"
METRIC = "chamfer_full"  # Lower-is-better error, or event_count_ratio (absolute distance from 1)
FOCUS = "evtween_v5.2_student"
EXPERIMENTS = {2:"windows_k2", 5:"windows_k5", 10:"windows_k10", 17:"windows_k17"}
STRIDE_REPORT_ID = None
WINDOW_REPORT_IDS = {}  # e.g. {17: "report folder id"}
BIN_MS = 10.0  # For window voxel reports; select a matching stride report when comparing protocols.
STEPS = [1, 2, 3, 4]
SEQUENCES = ["02_rocket_earth_light", "06_ziggy_and_fuzz", "10_office", "14_ziggy_in_the_arena"]
ERRORS = {"chamfer_full","chamfer_count","chamfer_penalty","chamfer","iei_distance","fano_factor",
          "voxel_mae","voxel_rmse","voxel_relative_l1","pmse_2_v2ce_adapted","pmse_4_v2ce_adapted","event_count_ratio"}
if METRIC not in ERRORS:
    raise ValueError("Select an error metric from " + str(sorted(ERRORS)))

def valid_rows(df, statuses):
    rows = df.loc[df.metric.eq(METRIC) & df.sequence.isin(SEQUENCES) & df.status.isin(statuses)
                  & df.method.ne("ground_truth") & np.isfinite(df.value)].copy()
    rows["error"] = (rows.value - 1).abs() if METRIC == "event_count_ratio" else rows.value
    return rows

def choose(paths, label):
    if not paths:
        return None
    candidates = []
    for path in paths:
        window = path.name == "per_window.csv"
        statuses = ["ok", "partial_support"] if window and METRIC == "fano_factor" else ["ok"]
        rows = valid_rows(pd.read_csv(path), statuses)
        if window:
            keys = ["sequence", "k", "first", "last"]
            focus = rows.loc[rows.method.eq(FOCUS), keys].drop_duplicates()
            other = rows.loc[rows.method.ne(FOCUS), keys + ["method"]].drop_duplicates()
            usable = other.merge(focus, on=keys)
        else:
            usable = rows[rows.stride.isin(STEPS)].groupby(["sequence", "method"]).filter(
                lambda group: group.stride.nunique() == len(STEPS))
        coverage = (len(usable), usable.sequence.nunique(), usable.method.nunique())
        candidates.append((coverage, path.stat().st_mtime_ns, path.parent.name, path))
    ranked = sorted(candidates, reverse=True)
    if len(ranked) > 1:
        for coverage, _, report_id, _ in ranked:
            print(f"{label}: {report_id} | usable rows={coverage[0]}, sequences={coverage[1]}, methods={coverage[2]}")
    selected = ranked[0][-1]
    print(f"Using {label} = {selected.parent.name}")
    return selected

unit_label = "|count ratio − 1|" if METRIC == "event_count_ratio" else METRIC

def show_chart(fig, name):
    from IPython.display import FileLink, HTML, display
    # Relative to the notebook working directory so Jupyter can serve the link.
    path = Path("compare_both_figures") / f"{name}_{METRIC}.html"
    path.parent.mkdir(exist_ok=True)
    fig.write_html(str(path), include_plotlyjs=True, auto_open=False)
    print(f"Interactive chart saved: {path.resolve()}")
    display(FileLink(path.as_posix()))
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))

def read_scores(path, statuses):
    df = pd.read_csv(path)
    selected = df[df.metric.eq(METRIC) & df.sequence.isin(SEQUENCES)]
    print(f"{path.parent.name}: {len(selected)} selected rows; statuses {selected.status.value_counts().to_dict()}")
    rows = valid_rows(df, statuses)
    print("Valid rows by method:", rows.method.value_counts().to_dict())
    return rows

## Stride · How much accuracy does each method lose?
Left: error level. Right: change from the same sequence at stride 1. Each method uses its sequences with valid scores at all four strides. Hover for coverage. Select a sequence to inspect it individually.

In [ ]:
base = REPORT / "full_sequence_strides"
candidates = []
for p in sorted(base.glob("*/per_sequence.csv")):
    if STRIDE_REPORT_ID and p.parent.name != STRIDE_REPORT_ID:
        continue
    if METRIC in pd.read_csv(p, usecols=["metric"]).metric.unique():
        candidates.append(p)
path = choose(candidates, "STRIDE_REPORT_ID")
if path is None:
    print(f"No saved stride report for {METRIC} in {base}")
else:
    data = read_scores(path, ["ok"])
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Error level", "Change from all frames"))
    choices = ["Mean", *SEQUENCES]
    views = []
    for view in choices:
        for i, (method, rows) in enumerate(data.groupby("method")):
            matrix = rows.pivot(index="sequence", columns="stride", values="error").reindex(columns=STEPS).dropna()
            if view != "Mean":
                matrix = matrix.loc[matrix.index == view]
            if matrix.empty:
                continue
            color = "#b33d28" if method == FOCUS else qualitative.Safe[i % len(qualitative.Safe)]
            for col, values in enumerate([matrix, matrix.sub(matrix[1], axis=0)], 1):
                fig.add_trace(go.Scatter(x=STEPS, y=values.mean().tolist(), name=method, legendgroup=method,
                    showlegend=col == 1, visible=view == "Mean", mode="lines+markers",
                    line=dict(color=color, width=4 if method == FOCUS else 2),
                    text=[", ".join(matrix.index)]*4,
                    hovertemplate="Stride %{x} · %{y:.4g}<br>%{text}<extra>%{fullData.name}</extra>"), row=1,col=col)
                views.append(view)
    fig.add_hline(y=0, line_dash="dot",row=1,col=2)
    fig.update_xaxes(tickvals=STEPS,ticktext=["1 · 100%","2 · ~50%","3 · ~33%","4 · ~25%"],title="Stride · frames retained")
    fig.update_yaxes(title_text=unit_label,row=1,col=1)
    fig.update_yaxes(title_text="Δ error (positive = worse)",row=1,col=2)
    fig.update_layout(template="plotly_white",height=470,margin=dict(t=110),
        title=f"{METRIC} | RGB subsampling", updatemenus=[dict(x=1,y=1.25,buttons=[
        dict(label=v,method="update",args=[{"visible":[a==v for a in views]}]) for v in choices])])
    if views:
        show_chart(fig, "stride")
    else:
        print("No method has valid scores at all four strides. Available strides:")
        print(data.groupby(["method","sequence"]).stride.unique().to_string())

## K · How often and by how much does Evtween beat a baseline?
Choose a baseline. Each dot represents scores on the same sequence and frame window. Positive differences favor Evtween. Diamonds mark medians. Box spans the middle 50%. Labels report wins / valid pairs, with ties retained in the denominator. Each K uses its own available windows.

In [ ]:
frames = []
for k, experiment in EXPERIMENTS.items():
    base = REPORT / experiment / f"k_{k}" / METRIC
    paths = []
    for p in sorted(base.glob("*/per_window.csv")):
        if k in WINDOW_REPORT_IDS and p.parent.name != WINDOW_REPORT_IDS[k]:
            continue
        cfg = json.loads((p.parent / "config.json").read_text())
        if cfg["start_stride"] != 50 or cfg["m"] != 0:
            continue
        if METRIC.startswith(("voxel_", "pmse_")) and cfg["options"].get("bin_ms") != BIN_MS:
            continue
        paths.append(p)
    p = choose(paths, f"WINDOW_REPORT_IDS[{k}]")
    if p is None:
        print(f"K={k}: no matching saved report in {base}")
        continue
    df = read_scores(p, ["ok","partial_support"] if METRIC=="fano_factor" else ["ok"])
    frames.append(df)
if frames:
    data = pd.concat(frames,ignore_index=True)
    keys = ["sequence","k","first","last"]
    focus = data.loc[data.method == FOCUS, keys+["error"]].rename(columns={"error":"focus"})
    fig = go.Figure()
    owners = []
    for method, group in data[data.method != FOCUS].groupby("method"):
        pairs = group.merge(focus, on=keys, validate="one_to_one")
        if pairs.empty:
            continue
        first_method = not owners or owners[0]==method
        for k, rows in pairs.groupby("k"):
            delta = rows.error - rows.focus
            x = list(EXPERIMENTS).index(k)
            fig.add_box(x=[x]*len(rows),y=delta.tolist(),width=.38,boxpoints=False,
                        fillcolor="rgba(53,108,160,.18)",line_color="#396d9e",visible=first_method,showlegend=False)
            jitter = np.random.default_rng(37).uniform(-.14,.14,len(rows))
            fig.add_scatter(x=x+jitter,y=delta.tolist(),mode="markers",visible=first_method,showlegend=False,
                marker=dict(size=6,opacity=.55,color="#396d9e"),
                customdata=rows[["sequence","first","last","error","focus"]].to_numpy(),
                hovertemplate="%{customdata[0]} · frames %{customdata[1]}–%{customdata[2]}<br>Baseline %{customdata[3]:.4g} · Evtween %{customdata[4]:.4g}<br>Difference %{y:.4g}<extra></extra>")
            fig.add_scatter(x=[x+.21],y=[delta.median()],mode="markers+text",visible=first_method,
                showlegend=False,marker=dict(symbol="diamond",color="#b33d28",size=10),
                text=[f"{(delta>0).sum()}/{len(delta)}"],textposition="top center",
                hovertemplate="Median %{y:.4g}<br>Wins %{text}<extra></extra>")
            owners.extend([method]*3)
    if owners:
        fig.add_hline(y=0,line_dash="dot")
        fig.update_layout(template="plotly_white",height=510,title=f"{FOCUS} | paired window differences",
            margin=dict(t=100),xaxis=dict(title="Input frames per window (K)",tickvals=list(range(len(EXPERIMENTS))),
            ticktext=list(map(str,EXPERIMENTS)),range=[-.5,len(EXPERIMENTS)-.5]),
            yaxis_title=f"Baseline − Evtween ({unit_label})",updatemenus=[dict(y=1.18,x=1,buttons=[
            dict(label=m,method="update",args=[{"visible":[o==m for o in owners]}]) for m in dict.fromkeys(owners)])])
        show_chart(fig, "windows")
    else:
        print(f"No valid matched windows for {FOCUS} and a baseline.")
else:
    print(f"No window reports matched. Check EXPERIMENTS against the directories under {REPORT}.")